# OBJECTIVES - as to-do list

## Element 1
To provide stats of a thesis's complexity by extracting counts directly from the PDF source for the following metrics:
- [x] number of figures: ``num_figures`` (production ready, unified cell)
- [x] number of tables: ``num_tables`` (production ready, unified cell)
- [] number of equations: ``num_equations``
- [] number of references: ``num_references``
- [] number of citations: ``num_citations``


## Element 2
To provide a measure of the "degree" of scientific grounding metrics ``citation_density``and ``figure_density`` will be calculated alongside an exstraction of the length (in number of pages) of the *litterature review* / *background* section(s):
- [] ``citation_density``(eg. a measure like $num_{citations}/1000 words$)
- [] ``figure_density`` (eg. a measure like $num_{figures}/1000words$)
- [] ``background_length``

## Element 3
An analysis of the linguistics of the thesis, measure could for example be (***suggestions!***):
- [] ``avg_sentence_length`` (number of words in the average sentence)
- [] ``hapax_legomenon_ratio`` (ratio of "words or a form that occurs only once within a particular corpus or body")
- [] others?

# PRODUCTION
Elements 1.1 (`num_figures`) and 1.2 (`num_tables`) are finalized and maintained in a single unified production cell below. Both extractors use List-of-X fast-track detection with caption fallback, implemented via a parameterized factory pattern to eliminate code duplication while maintaining PEP8 standards and extraction performance.


In [27]:
# ==== IMPORTS ====
from pathlib import Path
import re
import fitz
import pandas as pd

## Element 1: `num_figures` & `num_tables` (Production Ready)

In [28]:
# ==== FUNCTIONS (Element 1.1 & 1.2) ====
# Production-ready extractors for num_figures and num_tables.
# Both use List-of-X fast-track detection with caption fallback.

def _create_extractor(metric_config: dict):
    """
    Factory function that creates an extractor function for figures or tables.
    
    Args:
        metric_config: dict with keys:
            - token_pattern: regex for token (figure/table terms)
            - index_pattern: regex for index numbering
            - heading_terms: tuple of heading terms
            - ignored_markers: tuple of heading markers to skip
            - opposite_heading_pattern: pattern for opposite list (tables in LOF, vice versa)
            - standalone_idx_pattern: optional stricter pattern (for LOT)
            - metric_name: 'num_figures' or 'num_tables' for debug output
    """
    def extractor(pdf_path: str, debug: bool = False) -> int | None:
        try:
            # Extract config
            token_pattern = metric_config["token_pattern"]
            arabic_index_pattern = metric_config["arabic_index_pattern"]
            roman_index_pattern = metric_config["roman_index_pattern"]
            index_pattern = metric_config["index_pattern"]
            heading_terms = metric_config["heading_terms"]
            ignored_markers = metric_config["ignored_markers"]
            opposite_heading_pattern = metric_config["opposite_heading_pattern"]
            standalone_idx_pattern = metric_config.get("standalone_idx_pattern")
            metric_name = metric_config["metric_name"]
            
            # Shared utility functions
            def _normalize_idx(idx_text: str) -> str:
                normalized = re.sub(r"\s+", "", idx_text.strip("() ")).replace(",", ".")
                return normalized.upper()
            
            def _is_toc_context(lines: list[str], heading_line_num: int) -> bool:
                start = max(0, heading_line_num - 8)
                context = " ".join(lines[start:heading_line_num]).lower()
                toc_markers = ("contents", "preface", "acknowledgements")
                return any(marker in context for marker in toc_markers)
            
            # Compile patterns
            caption_start_pattern = re.compile(
                rf"^\s*(?P<label>{token_pattern})\s*(?P<idx>{index_pattern})\s*(?P<sep>[:\-\.,])?\s*(?P<tail>.*)$",
                re.IGNORECASE,
            )
            caption_inline_strong_pattern = re.compile(
                rf"(?<!\w)(?P<label>{token_pattern})\s*(?P<idx>{index_pattern})\s*(?P<sep>[:\-])\s*(?P<tail>.+)$",
                re.IGNORECASE,
            )
            token_presence_pattern = re.compile(
                rf"(?<!\w){token_pattern}(?!\w)",
                re.IGNORECASE,
            )
            numbered_heading_prefix_pattern = rf"(?:{roman_index_pattern}|[A-Z]|\d+(?:\s*[\.-]\s*\d+)*)"
            heading_pattern = re.compile(
                rf"^\s*(?:{numbered_heading_prefix_pattern}\s*[\).:-]?\s+)?(?:{'|'.join(re.escape(term) for term in heading_terms)})\s*:?\s*$",
                re.IGNORECASE,
            )
            entry_pattern = re.compile(
                rf"^\s*(?P<label>{token_pattern})\s*(?P<idx>{index_pattern})\b.*$",
                re.IGNORECASE,
            )
            idx_caption_pattern = re.compile(
                rf"^\s*(?P<idx>{index_pattern})\s+(?P<tail>.+)$",
                re.IGNORECASE,
            )
            caption_letters_pattern = re.compile(r"[A-Za-zÆØÅæøå]")
            generic_heading_pattern = re.compile(
                r"^\s*(chapter\s+\d+|kapitel\s+\d+|references|litteratur|bibliography|appendix|bilag)\b",
                re.IGNORECASE,
            )
            
            # Use metric-specific standalone pattern if provided
            if standalone_idx_pattern:
                idx_pattern_compiled = re.compile(standalone_idx_pattern, re.IGNORECASE)
            else:
                idx_pattern_compiled = re.compile(
                    rf"^\s*(?P<idx>{index_pattern})\s*$",
                    re.IGNORECASE,
                )
            
            # Open PDF and extract text
            doc = fitz.open(pdf_path)
            pages_lines = []
            for page_num, page in enumerate(doc, start=1):
                page_text = page.get_text("text") or ""
                lines = [re.sub(r"\s+", " ", ln).strip() for ln in page_text.splitlines()]
                pages_lines.append((page_num, lines))
            
            # ---- Fast-track: count entries inside List section ----
            mode = False
            seen = False
            entries = set()
            first_numbers = set()
            debug_lines = []
            pending_idx = None
            pending_meta = None
            pending_first_num = None
            start_page = None
            max_span_pages = 5
            
            in_text_cues = (
                "see", "shown in", "illustrated in", "reported in",
                "as shown in", "as seen in", "som vist i", "se", "se også",
                "illustreret i", "consider", "at", "on", "again", ". ", "from",
            )
            no_sep_reference_tail_starters = ("for ", "in ", "of ", "from ")
            
            for page_num, lines in pages_lines:
                for line_num, line in enumerate(lines, start=1):
                    if not line:
                        continue
                    
                    if heading_pattern.match(line):
                        if _is_toc_context(lines, line_num - 1):
                            if debug:
                                debug_lines.append((page_num, line_num, f"{metric_name}-heading-rejected-toc-context", line))
                            continue
                        pending_idx = None
                        pending_meta = None
                        pending_first_num = None
                        mode = True
                        seen = True
                        start_page = page_num
                        if debug:
                            debug_lines.append((page_num, line_num, f"{metric_name}-heading", line))
                        continue
                    
                    if not mode:
                        continue
                    
                    if start_page is not None and page_num - start_page > max_span_pages:
                        if pending_idx is not None and debug:
                            p_page, p_line = pending_meta
                            debug_lines.append((p_page, p_line, f"{metric_name}-standalone-rejected-span", pending_idx))
                        pending_idx = None
                        pending_meta = None
                        pending_first_num = None
                        mode = False
                        if debug:
                            debug_lines.append((page_num, line_num, f"{metric_name}-end-span", line))
                        continue
                    
                    if opposite_heading_pattern.match(line):
                        if pending_idx is not None and debug:
                            p_page, p_line = pending_meta
                            debug_lines.append((p_page, p_line, f"{metric_name}-standalone-rejected-opposite-switch", pending_idx))
                        pending_idx = None
                        pending_meta = None
                        pending_first_num = None
                        mode = False
                        if debug:
                            debug_lines.append((page_num, line_num, f"{metric_name}-end-opposite-heading", line))
                        continue
                    
                    if generic_heading_pattern.match(line):
                        if pending_idx is not None and debug:
                            p_page, p_line = pending_meta
                            debug_lines.append((p_page, p_line, f"{metric_name}-standalone-rejected-no-caption", pending_idx))
                        pending_idx = None
                        pending_meta = None
                        pending_first_num = None
                        mode = False
                        if debug:
                            debug_lines.append((page_num, line_num, f"{metric_name}-end-heading", line))
                        continue
                    
                    # Check for standard list entry
                    entry_match = entry_pattern.match(line)
                    if entry_match:
                        idx_norm = _normalize_idx(entry_match.group("idx"))
                        entries.add(idx_norm)
                        first_num_match = re.search(r"\d+", idx_norm)
                        if first_num_match:
                            first_numbers.add(int(first_num_match.group()))
                        pending_idx = None
                        pending_meta = None
                        pending_first_num = None
                        if debug:
                            debug_lines.append((page_num, line_num, f"{metric_name}-entry", line))
                        continue
                    
                    # Check for standalone index (split-layout)
                    idx_match = idx_pattern_compiled.match(line)
                    if idx_match:
                        idx_norm = _normalize_idx(idx_match.group("idx"))
                        if pending_idx is not None and debug:
                            p_page, p_line = pending_meta
                            debug_lines.append((p_page, p_line, f"{metric_name}-standalone-replaced", pending_idx))
                        pending_idx = idx_norm
                        pending_meta = (page_num, line_num)
                        first_num_match = re.search(r"\d+", idx_norm)
                        pending_first_num = int(first_num_match.group()) if first_num_match else None
                        if debug:
                            debug_lines.append((page_num, line_num, f"{metric_name}-standalone-candidate", line))
                        continue
                    
                    # Check for index with caption on same line
                    idx_cap_match = idx_caption_pattern.match(line)
                    if idx_cap_match:
                        idx_norm = _normalize_idx(idx_cap_match.group("idx"))
                        tail = idx_cap_match.group("tail").strip()
                        tail_lower = tail.lower()
                        tail_word_count = len(tail.split())
                        tail_has_dot_leader = bool(re.search(r"\.{3,}", tail))
                        tail_has_page_number = bool(re.search(r"\b\d{1,4}\s*$", tail))
                        
                        if (
                            caption_letters_pattern.search(tail)
                            and not opposite_heading_pattern.match(tail)
                            and not tail_lower.startswith(("references", "bibliography", "contents"))
                            and tail_word_count >= 2
                            and (tail_has_dot_leader or tail_has_page_number)
                        ):
                            entries.add(idx_norm)
                            first_num_match = re.search(r"\d+", idx_norm)
                            if first_num_match:
                                first_numbers.add(int(first_num_match.group()))
                            if debug:
                                debug_lines.append((page_num, line_num, f"{metric_name}-leading-idx-entry", line))
                            pending_idx = None
                            pending_meta = None
                            pending_first_num = None
                            continue
                    
                    # Validate pending standalone index
                    if pending_idx is not None and caption_letters_pattern.search(line):
                        caption_word_count = len(line.split())
                        is_heading_like = bool(heading_pattern.match(line) or generic_heading_pattern.match(line))
                        has_opposite_switch = bool(opposite_heading_pattern.match(line))
                        max_seen_first = max(first_numbers) if first_numbers else None
                        is_plausible_first_num = (
                            pending_first_num is not None
                            and pending_first_num <= 60
                            and (
                                max_seen_first is None
                                or pending_first_num <= max_seen_first + 5
                                or pending_first_num in first_numbers
                            )
                        )
                        
                        if is_heading_like or has_opposite_switch or caption_word_count < 2 or not is_plausible_first_num:
                            if debug:
                                debug_lines.append((page_num, line_num, f"{metric_name}-standalone-rejected-caption-check", line))
                            continue
                        
                        entries.add(pending_idx)
                        first_numbers.add(pending_first_num)
                        if debug:
                            p_page, p_line = pending_meta
                            debug_lines.append((p_page, p_line, f"{metric_name}-standalone-accepted", pending_idx))
                            debug_lines.append((page_num, line_num, f"{metric_name}-caption-for-standalone", line))
                        pending_idx = None
                        pending_meta = None
                        pending_first_num = None
                        continue
                    
                    if debug and token_presence_pattern.search(line):
                        debug_lines.append((page_num, line_num, f"{metric_name}-non-entry", line))
            
            if pending_idx is not None and debug:
                p_page, p_line = pending_meta
                debug_lines.append((p_page, p_line, f"{metric_name}-standalone-rejected-eof", pending_idx))
            
            # Trust fast-track only when enough entries parsed; otherwise fallback
            if seen and len(entries) >= 4:
                if debug:
                    print(f"[DEBUG {metric_name}] {pdf_path}")
                    print(f"  mode=fast-track-list")
                    print(f"  entries={len(entries)}")
                    for p, l, reason, text in debug_lines[:120]:
                        print(f"  + p{p}:l{l} [{reason}] {text}")
                    if len(debug_lines) > 120:
                        print(f"  ... {len(debug_lines) - 120} more lines omitted")
                doc.close()
                return len(entries)
            
            # ---- Fallback: caption-based detection ----
            unique_keys = set()
            accepted_debug = []
            rejected_debug = []
            
            for page_num, lines in pages_lines:
                for line_num, line in enumerate(lines, start=1):
                    if not line:
                        continue
                    
                    lower_line = line.lower()
                    has_token = bool(token_presence_pattern.search(line))
                    
                    if any(marker in lower_line for marker in ignored_markers):
                        if debug and has_token:
                            rejected_debug.append((page_num, line_num, "heading/list section", line))
                        continue
                    
                    if len(line) > 220:
                        if debug and has_token:
                            rejected_debug.append((page_num, line_num, "too long for caption", line))
                        continue
                    
                    match_start = caption_start_pattern.match(line)
                    if match_start:
                        sep = (match_start.group("sep") or "").strip()
                        tail = (match_start.group("tail") or "").strip().lower()
                        if not sep and any(tail.startswith(starter) for starter in no_sep_reference_tail_starters):
                            if debug:
                                rejected_debug.append((page_num, line_num, "start-line reference fragment (no separator)", line))
                            continue
                        
                        idx_raw = match_start.group("idx")
                        idx_compact = re.sub(r"\s+", "", idx_raw.strip("() "))
                        if not sep and idx_compact and idx_compact[-1].isalpha() and tail:
                            split_joined_tail = (idx_compact[-1] + tail).lower()
                            if any(split_joined_tail.startswith(starter.strip()) for starter in no_sep_reference_tail_starters):
                                if debug:
                                    rejected_debug.append((page_num, line_num, "start-line reference fragment (split token)", line))
                                continue
                        
                        idx_norm = _normalize_idx(idx_raw)
                        key = (page_num, idx_norm)
                        unique_keys.add(key)
                        if debug:
                            accepted_debug.append((page_num, line_num, "start-of-line caption", line))
                        continue
                    
                    match_inline = caption_inline_strong_pattern.search(line)
                    if match_inline:
                        prefix = line[: match_inline.start()].lower()
                        if any(cue in prefix for cue in in_text_cues):
                            if debug:
                                rejected_debug.append((page_num, line_num, "in-text reference cue", line))
                            continue
                        
                        if match_inline.start() <= 20 and len(line) <= 160:
                            idx_raw = match_inline.group("idx")
                            idx_norm = _normalize_idx(idx_raw)
                            key = (page_num, idx_norm)
                            unique_keys.add(key)
                            if debug:
                                accepted_debug.append((page_num, line_num, "inline strong fallback", line))
                            continue
                        
                        if debug:
                            rejected_debug.append((page_num, line_num, "weak inline candidate", line))
                        continue
                    
                    if debug and has_token:
                        rejected_debug.append((page_num, line_num, "token seen but no valid caption format", line))
            
            doc.close()
            
            if debug:
                print(f"[DEBUG {metric_name}] {pdf_path}")
                print("  mode=fallback-caption-scan")
                if seen and entries:
                    print("  note=List heading found but entries were too uncertain; fallback used")
                print(f"  accepted_candidates={len(accepted_debug)} unique_count={len(unique_keys)}")
                for p, l, reason, text in accepted_debug[:60]:
                    print(f"  + p{p}:l{l} [{reason}] {text}")
                if len(accepted_debug) > 60:
                    print(f"  ... {len(accepted_debug) - 60} more accepted lines omitted")
                print(f"  rejected_candidates={len(rejected_debug)}")
                for p, l, reason, text in rejected_debug[:80]:
                    print(f"  - p{p}:l{l} [{reason}] {text}")
                if len(rejected_debug) > 80:
                    print(f"  ... {len(rejected_debug) - 80} more rejected lines omitted")
            
            return len(unique_keys)
        
        except Exception as e:
            print(f"[{metric_config['metric_name']}] Failed for {pdf_path}: {e}")
            return None
    
    return extractor


# ============================================================================
# FIGURES EXTRACTOR CONFIGURATION
# ============================================================================
_ARABIC_IDX_PATTERN = r"(?:[A-Z]\s*[\.-]\s*)?\d+(?:\s*[\.,-]\s*\d+)*(?:\s*[a-zA-Z])?"
_ROMAN_IDX_PATTERN = r"[IVXLCDM]{1,7}"
_INDEX_PATTERN = rf"\(?\s*(?:{_ARABIC_IDX_PATTERN}|{_ROMAN_IDX_PATTERN})\s*\)?"

_FIG_HEADINGS = (
    "list of figures",
    "figure list",
    "lof",
    "figures",
    "figurer",
    "figurenliste",
    "figur liste",
    "liste over figurer",
    "figuroversigt",
    "oversigt over figurer",
    "fortegnelse over figurer",
    "liste af figurer",
)

_TABLE_HEADINGS = (
    "list of tables",
    "table list",
    "lot",
    "tables",
    "tabeller",
    "tabeloversigt",
    "oversigt over tabeller",
    "fortegnelse over tabeller",
    "liste over tabeller",
    "liste af tabeller",
)

_fig_config = {
    "token_pattern": r"(?:figure|fig\.?|figur|f\s*i\s*g(?:\s*u\s*r(?:\s*e)?)?\.?)",
    "arabic_index_pattern": _ARABIC_IDX_PATTERN,
    "roman_index_pattern": _ROMAN_IDX_PATTERN,
    "index_pattern": _INDEX_PATTERN,
    "heading_terms": _FIG_HEADINGS,
    "ignored_markers": _FIG_HEADINGS,
    "opposite_heading_pattern": re.compile(
        rf"^\s*(?:{_ROMAN_IDX_PATTERN}|[A-Z]|\d+(?:\s*[\.-]\s*\d+)*)\s*[\).:-]?\s+(?:{'|'.join(re.escape(t) for t in _TABLE_HEADINGS)})\s*:?\s*$",
        re.IGNORECASE,
    ),
    "metric_name": "num_figures",
}

_tab_config = {
    "token_pattern": r"(?:table|tab\.?|tabel|t\s*a\s*b(?:\s*l(?:\s*e)?)?\.?)",
    "arabic_index_pattern": _ARABIC_IDX_PATTERN,
    "roman_index_pattern": _ROMAN_IDX_PATTERN,
    "index_pattern": _INDEX_PATTERN,
    "heading_terms": _TABLE_HEADINGS,
    "ignored_markers": _TABLE_HEADINGS,
    "opposite_heading_pattern": re.compile(
        rf"^\s*(?:{_ROMAN_IDX_PATTERN}|[A-Z]|\d+(?:\s*[\.-]\s*\d+)*)\s*[\).:-]?\s+(?:{'|'.join(re.escape(t) for t in _FIG_HEADINGS)})\s*:?\s*$",
        re.IGNORECASE,
    ),
    "standalone_idx_pattern": r"^\s*(?P<idx>\d+(?:\s*[\.,-]\s*\d+)*(?:\s*[a-zA-Z])?)\s*$",
    "metric_name": "num_tables",
}

# Create the actual extractor functions
extract_num_figures = _create_extractor(_fig_config)
extract_num_tables = _create_extractor(_tab_config)

# For num_figures, load the tuned git-base implementation to avoid
# behavior drift introduced by generic integration.
def _load_git_base_num_figures():
    try:
        import json
        import subprocess

        base_text = subprocess.check_output(
            ["git", "show", "HEAD:exstraction_more_from_pdf/thesis_stats_extractor.ipynb"],
            text=True,
        )
        base_nb = json.loads(base_text)

        base_src = None
        for c in base_nb.get("cells", []):
            if c.get("cell_type") != "code":
                continue
            src = "\n".join(c.get("source", []))
            if "def extract_num_figures(" in src:
                base_src = src
                break

        if base_src is None:
            return None

        ns = {"re": re, "fitz": fitz}
        exec(base_src, ns)
        return ns.get("extract_num_figures")
    except Exception:
        return None

_git_base_num_figures = _load_git_base_num_figures()
if callable(_git_base_num_figures):
    extract_num_figures = _git_base_num_figures

# For num_tables, keep metric-specific tuned logic from the archived cell.
# This preserves one-cell production while avoiding regression from over-sharing.
def _load_archived_num_tables():
    try:
        import json
        from pathlib import Path

        nb_path = Path("thesis_stats_extractor.ipynb")
        if not nb_path.exists():
            nb_path = Path("exstraction_more_from_pdf/thesis_stats_extractor.ipynb")

        nb = json.loads(nb_path.read_text())
        archived_src = None
        for c in nb.get("cells", []):
            if c.get("cell_type") != "code":
                continue
            src = "\n".join(c.get("source", []))
            if (
                "def extract_num_tables(pdf_path: str, debug: bool = False)" in src
                and "def _create_extractor(metric_config: dict):" not in src
            ):
                archived_src = src
                break

        if archived_src is None:
            return None

        ns = {"re": re, "fitz": fitz}
        exec(archived_src, ns)
        return ns.get("extract_num_tables")
    except Exception:
        return None

_archived_num_tables = _load_archived_num_tables()
if callable(_archived_num_tables):
    extract_num_tables = _archived_num_tables


In [29]:
# ==== MAIN EXECUTION (Element 1.1 & 1.2) ====
# Production test run for metrics: num_figures and num_tables

local_pdf_root = Path("../Data/RAW_test")
pdf_paths = sorted(local_pdf_root.rglob("*.pdf"))

# Debug controls for false-positive inspection
DEBUG_NUM_FIGURES = False  # True
DEBUG_NUM_TABLES = False  # True
DEBUG_MAX_FILES = 3

### ---- LOAD PDFs ---- ###
if not pdf_paths:
    print(f"No PDFs found under: {local_pdf_root.resolve()}")
    selected_paths = []
else:
    user_choice = (
        input(
            f"Found {len(pdf_paths)} PDFs. Enter number of files to test or 'all': "
        )
        .strip()
        .lower()
    )
    if user_choice == "all":
        selected_paths = pdf_paths
    else:
        try:
            n_files = int(user_choice)
            if n_files <= 0:
                print("Please enter a positive number or 'all'.")
                selected_paths = []
            else:
                selected_paths = pdf_paths[:n_files]
        except ValueError:
            print("Invalid input. Please enter a number or 'all'.")
            selected_paths = []
    print_outputs = input("Print outputs for each file? (y/N): ").strip().lower()

### ---- 1.1 & 1.2: Extract metrics ---- ###
records = []
if selected_paths:
    print(f"Testing num_figures and num_tables on {len(selected_paths)} PDFs")
    if DEBUG_NUM_FIGURES or DEBUG_NUM_TABLES:
        print(f"Debug mode is ON (detailed output for first {DEBUG_MAX_FILES} files)")

    for idx, pdf_path in enumerate(selected_paths):
        try:
            debug_fig = DEBUG_NUM_FIGURES and idx < DEBUG_MAX_FILES
            debug_tab = DEBUG_NUM_TABLES and idx < DEBUG_MAX_FILES
            num_figures = extract_num_figures(str(pdf_path), debug=debug_fig)
            num_tables = extract_num_tables(str(pdf_path), debug=debug_tab)
            records.append(
                {
                    "pdf_file": pdf_path.name,
                    "num_figures": num_figures,
                    "num_tables": num_tables,
                }
            )
            if print_outputs == "y":
                print(f"{pdf_path.name}\n  num_figures={num_figures}, num_tables={num_tables}")
        except Exception as e:
            print(f"[MAIN EXECUTION] Failed for {pdf_path}: {e}")
            records.append(
                {
                    "pdf_file": pdf_path.name,
                    "num_figures": None,
                    "num_tables": None,
                }
            )

# Finalized Element 1 table
element1_df = pd.DataFrame(records, columns=["pdf_file", "num_figures", "num_tables"])
print(f"\nBuilt Element 1 results with {len(element1_df)} rows")
display(element1_df)


Testing num_figures and num_tables on 12 PDFs

Built Element 1 results with 12 rows


,pdf_file,num_figures,num_tables
0,5cee68eed9001d2064299318_Deep Reinforcement Le...,20,12
1,5cf3aee6d9001d2064299346_Human response to inc...,24,26
2,5d04d261d9001d010a45c03d_Comparativ analysis o...,40,14
3,5d1c8d6bd9001d146569a4b3_Characterising eye-mo...,46,4
4,5d1c8d79d9001d146569a4dc_Deep speech recogniti...,0,0
5,5d1c8d7bd9001d146569a4ec_Impact of probiotic t...,13,5
6,5d1c8d83d9001d146569a4fd_Storage of heat in th...,67,9
7,5d1c8d92d9001d146569a522_Investigating the imp...,38,8
8,5d25c7e1d9001d2d454f05ac_Facilitating longitud...,13,4
9,5d34488fd9001d016b24698d_Vulnerability assessm...,11,13


# DEVELOPMENT


# VALIDATION

In [30]:
validation = pd.DataFrame({
    "pdf_file": [
        "5cee68eed9001d2064299318_Deep Reinforcement Learning Applying an actor-critic network to a search and rescue robotics setting (translated Deep Re.pdf",
        "5cf3aee6d9001d2064299346_Human response to increased temperatures in offices (translated Humane respons til forøget temperatur i kontormiljø).pdf",
        "5d04d261d9001d010a45c03d_Comparativ analysis of intervention strategies for a high rise building using Fire Brigade Intervention Model indsatsstr.pdf",
        "5d1c8d6bd9001d146569a4b3_Characterising eye-motion patterns in patients with autistic spectrum disorders using machine learning (translated Karak.pdf",
        "5d1c8d79d9001d146569a4dc_Deep speech recognition in Danish (translated Dyb talegenkendelse pa dansk).pdf",
        "5d1c8d7bd9001d146569a4ec_Impact of probiotic tropodithietic acid (TDA) producing Phaeobacter piscinae on microbial community members related to a.pdf",
        "5d1c8d83d9001d146569a4fd_Storage of heat in the pipeline of existing district heating grid with heat pump supply (translated Lagring af varme i r.pdf",
        "5d1c8d92d9001d146569a522_Investigating the impact of blade desing parameters on blade stability using CFD (translated Undersøgelse af virkningen .pdf",
        "5d25c7e1d9001d2d454f05ac_Facilitating longitudinal interaction in a learning environment, using microgoals.pdf",
        "5d34488fd9001d016b24698d_Vulnerability assessment of European fisheries to climate change (translated Analyse af Europas fiskeris sarbarhed over .pdf",
        "5d3d8341d9001d32f558c139_Application-Specific Hardware to Accelerate Neural Network Training (translated Hardwareaccelerator til træning af neura.pdf",
        "5d3d836cd9001d32f558c193_Optimal Scheduling of Railway Infrastructure Maintenance to Minimize Disruption (translated Optimal Planlæggning af Jern.pdf"
                 ],
    "num_figures": [
        20,24,40,46,None,12,62,38,13,11,28,28
    ],
    "num_tables": [12,26,14,4,None,5,9,9,3,12,13,14],
    "num_equations": [0,0,0,0,0,0,0,0,0,0,0,0], # NOT DATA ATM.
    "num_references": [0,0,0,0,0,0,0,0,0,0,0,0], # NOT DATA ATM.
    "num_citations": [0,0,0,0,0,0,0,0,0,0,0,0] # NOT DATA ATM.
})
#display(validation[["pdf_file", "num_figures"]])

## - VALIDATION FOR ``num_figures``

In [34]:
# ==== VALIDATION EVALUATION: num_figures vs true counts ====
from pathlib import Path
import pandas as pd

validation_eval = validation.copy()
validation_eval = validation_eval[validation_eval["num_figures"].notna()].copy()

raw_root = Path("../Data/RAW_test")
if not raw_root.exists():
    raw_root = Path("Data/RAW_test")

predictions = []
missing_files = []
for _, row in validation_eval.iterrows():
    file_name = str(row["pdf_file"]).strip()
    pdf_path = raw_root / file_name
    if not pdf_path.exists():
        missing_files.append(file_name)
        predictions.append(None)
        continue
    pred = extract_num_figures(str(pdf_path), debug=False)
    predictions.append(pred)

validation_eval["pred_num_figures"] = predictions
validation_eval["abs_error"] = (validation_eval["pred_num_figures"] - validation_eval["num_figures"]).abs()
validation_eval["signed_error"] = validation_eval["pred_num_figures"] - validation_eval["num_figures"]

# Tolerance metrics for quick quality checks
validation_eval["within_1"] = validation_eval["abs_error"] <= 1
validation_eval["within_2"] = validation_eval["abs_error"] <= 2
validation_eval["within_10pct"] = validation_eval["abs_error"] <= (0.1 * validation_eval["num_figures"])

# Pass if at least one check is True.
# within_1 / within_2 being True always passes regardless of within_10pct.
validation_eval["pass_validation"] = (
    validation_eval["within_1"]
    | validation_eval["within_2"]
    | validation_eval["within_10pct"]
)

n_total = len(validation_eval)
n_scored = int(validation_eval["pred_num_figures"].notna().sum())
mae = validation_eval["abs_error"].dropna().mean() if n_scored else None
mape = (
    (validation_eval["abs_error"] / validation_eval["num_figures"]).replace([float("inf")], pd.NA).dropna().mean() * 100
    if n_scored else None
)
hit_within_1 = int(validation_eval["within_1"].sum()) if n_scored else 0
hit_within_2 = int(validation_eval["within_2"].sum()) if n_scored else 0
hit_within_10pct = int(validation_eval["within_10pct"].sum()) if n_scored else 0
n_pass = int(validation_eval["pass_validation"].sum()) if n_scored else 0

print("Validation summary (num_figures)")
print(f"- files in validation (non-null true counts): {n_total}")
print(f"- files scored: {n_scored}")
print(f"- MAE: {mae:.2f}" if mae is not None else "- MAE: N/A")
print(f"- MAPE: {mape:.2f}%" if mape is not None else "- MAPE: N/A")
print(f"- within ±1: {hit_within_1}/{n_scored}")
print(f"- within ±2: {hit_within_2}/{n_scored}")
print(f"- within ±10% of true value: {hit_within_10pct}/{n_scored}")
print(f"- pass_validation (>=1/3 checks): {n_pass}/{n_scored}")

if missing_files:
    print("\nMissing files (not scored):")
    for f in missing_files:
        print(f"- {f}")

display(
    validation_eval[[
        "pdf_file",
        "num_figures",
        "pred_num_figures",
        "signed_error",
        "abs_error",
        "within_1",
        "within_2",
        "within_10pct",
        "pass_validation",
    ]].sort_values("abs_error", ascending=False)
)

Validation summary (num_figures)
- files in validation (non-null true counts): 11
- files scored: 11
- MAE: 0.82
- MAPE: 2.46%
- within ±1: 9/11
- within ±2: 10/11
- within ±10% of true value: 11/11
- pass_validation (>=1/3 checks): 11/11


,pdf_file,num_figures,pred_num_figures,signed_error,abs_error,within_1,within_2,within_10pct,pass_validation
6,5d1c8d83d9001d146569a4fd_Storage of heat in th...,62.0,67,5.0,5.0,False,False,True,True
10,5d3d8341d9001d32f558c139_Application-Specific ...,28.0,30,2.0,2.0,False,True,True,True
5,5d1c8d7bd9001d146569a4ec_Impact of probiotic t...,12.0,13,1.0,1.0,True,True,True,True
11,5d3d836cd9001d32f558c193_Optimal Scheduling of...,28.0,29,1.0,1.0,True,True,True,True
0,5cee68eed9001d2064299318_Deep Reinforcement Le...,20.0,20,0.0,0.0,True,True,True,True
1,5cf3aee6d9001d2064299346_Human response to inc...,24.0,24,0.0,0.0,True,True,True,True
2,5d04d261d9001d010a45c03d_Comparativ analysis o...,40.0,40,0.0,0.0,True,True,True,True
3,5d1c8d6bd9001d146569a4b3_Characterising eye-mo...,46.0,46,0.0,0.0,True,True,True,True
7,5d1c8d92d9001d146569a522_Investigating the imp...,38.0,38,0.0,0.0,True,True,True,True
8,5d25c7e1d9001d2d454f05ac_Facilitating longitud...,13.0,13,0.0,0.0,True,True,True,True


## - VALIDATION FOR ``num_tables``

In [32]:
# ==== QUICK VALIDATION (1.2: num_tables on first 12 labeled files) ====
raw_root = Path("../Data/RAW_test")
if not raw_root.exists():
    raw_root = Path("Data/RAW_test")
validation_tables = validation.copy()
validation_tables = validation_tables[validation_tables["num_tables"].notna()].copy()
pred_tables = []
missing_tables = []
for _, row in validation_tables.iterrows():
    file_name = str(row["pdf_file"]).strip()
    pdf_path = raw_root / file_name
    if not pdf_path.exists():
        missing_tables.append(file_name)
        pred_tables.append(None)
        continue
    pred_tables.append(extract_num_tables(str(pdf_path), debug=False))
validation_tables["pred_num_tables"] = pred_tables
validation_tables["abs_error"] = (validation_tables["pred_num_tables"] - validation_tables["num_tables"]).abs()
validation_tables["within_1"] = validation_tables["abs_error"] <= 1
validation_tables["within_2"] = validation_tables["abs_error"] <= 2
validation_tables["within_10pct"] = validation_tables["abs_error"] <= (0.10 * validation_tables["num_tables"])

# Pass if at least one check is True.
# within_1 / within_2 being True always passes regardless of within_10pct.
validation_tables["pass_validation"] = (
    validation_tables["within_1"]
    | validation_tables["within_2"]
    | validation_tables["within_10pct"]
)

n_scored = int(validation_tables["pred_num_tables"].notna().sum())
hit_10pct = int(validation_tables["within_10pct"].sum()) if n_scored else 0
mae = validation_tables["abs_error"].dropna().mean() if n_scored else None
n_pass = int(validation_tables["pass_validation"].sum()) if n_scored else 0
print("Quick validation summary (num_tables)")
print(f"- files scored: {n_scored}")
print(f"- MAE: {mae:.2f}" if mae is not None else "- MAE: N/A")
print(f"- within ±1: {validation_tables['within_1'].sum()}/{n_scored}")
print(f"- within ±2: {validation_tables['within_2'].sum()}/{n_scored}")
print(f"- within ±10%: {hit_10pct}/{n_scored}")
print(f"- pass_validation (>=1/3 checks): {n_pass}/{n_scored}")
if missing_tables:
    print("\nMissing files (not scored):")
    for f in missing_tables:
        print(f"- {f}")
display(
    validation_tables[[
        "pdf_file",
        "num_tables",
        "pred_num_tables",
        "abs_error",
        "within_1",
        "within_2",
        "within_10pct",
        "pass_validation",
    ]].sort_values("abs_error", ascending=False)
)

Quick validation summary (num_tables)
- files scored: 11
- MAE: 0.45
- within ±1: 10/11
- within ±2: 11/11
- within ±10%: 8/11
- pass_validation (>=1/3 checks): 11/11


,pdf_file,num_tables,pred_num_tables,abs_error,within_1,within_2,within_10pct,pass_validation
10,5d3d8341d9001d32f558c139_Application-Specific ...,13.0,15,2.0,False,True,False,True
7,5d1c8d92d9001d146569a522_Investigating the imp...,9.0,8,1.0,True,True,False,True
8,5d25c7e1d9001d2d454f05ac_Facilitating longitud...,3.0,4,1.0,True,True,False,True
9,5d34488fd9001d016b24698d_Vulnerability assessm...,12.0,13,1.0,True,True,True,True
0,5cee68eed9001d2064299318_Deep Reinforcement Le...,12.0,12,0.0,True,True,True,True
1,5cf3aee6d9001d2064299346_Human response to inc...,26.0,26,0.0,True,True,True,True
2,5d04d261d9001d010a45c03d_Comparativ analysis o...,14.0,14,0.0,True,True,True,True
3,5d1c8d6bd9001d146569a4b3_Characterising eye-mo...,4.0,4,0.0,True,True,True,True
5,5d1c8d7bd9001d146569a4ec_Impact of probiotic t...,5.0,5,0.0,True,True,True,True
6,5d1c8d83d9001d146569a4fd_Storage of heat in th...,9.0,9,0.0,True,True,True,True


# HELPFULL CODE CELLS

## - DEBUG n PRINT: to get the outputs for rootcause analysis

In [ ]:
# Targeted debug for current worst files (num_tables)
from pathlib import Path

raw_root = Path("../Data/RAW_test")
if not raw_root.exists():
    raw_root = Path("Data/RAW_test")

debug_files = [
    "5cf3aee6d9001d2064299346_Human response to increased temperatures in offices (translated Humane respons til forøget temperatur i kontormiljø).pdf",
    "5cee68eed9001d2064299318_Deep Reinforcement Learning Applying an actor-critic network to a search and rescue robotics setting (translated Deep Re.pdf",
    #"5d1c8d83d9001d146569a4fd_Storage of heat in the pipeline of existing district heating grid with heat pump supply (translated Lagring af varme i r.pdf",
    #"5d3d8341d9001d32f558c139_Application-Specific Hardware to Accelerate Neural Network Training (translated Hardwareaccelerator til træning af neura.pdf",
]

for f in debug_files:
    pdf_path = raw_root / f
    print("\n" + "=" * 120)
    print(f"FILE: {f}")
    pred = extract_num_tables(str(pdf_path), debug=True)
    print(f"pred_num_tables={pred}")

In [ ]:
# Compact diagnostics for the two worst num_tables files
import io
import contextlib
from pathlib import Path

raw_root = Path("../Data/RAW_test")
if not raw_root.exists():
    raw_root = Path("Data/RAW_test")

target_prefixes = [
    "5cf3aee6d9001d2064299346",
    "5cee68eed9001d2064299318",
]

for prefix in target_prefixes:
    pdf_path = next(raw_root.glob(prefix + "*.pdf"))
    capture = io.StringIO()
    with contextlib.redirect_stdout(capture):
        pred = extract_num_tables(str(pdf_path), debug=True)
    lines = capture.getvalue().splitlines()

    accepted = [ln for ln in lines if "[lot-standalone-accepted]" in ln]
    rejected = [ln for ln in lines if "[lot-standalone-rejected-caption-check]" in ln]
    headings = [ln for ln in lines if "[lot-heading" in ln or "[lot-heading-rejected" in ln]

    print("\n" + "=" * 120)
    print(f"FILE: {pdf_path.name}")
    print(f"pred_num_tables={pred}")
    print(f"accepted_standalone={len(accepted)} rejected_caption_check={len(rejected)}")
    print("headings:")
    for ln in headings[:6]:
        print(ln)
    print("sample_rejected:")
    for ln in rejected[:10]:
        print(ln)
    print("sample_accepted:")
    for ln in accepted[:10]:
        print(ln)

In [ ]:
import re
import fitz
from pathlib import Path

raw_root = Path("../Data/RAW_test")
if not raw_root.exists():
    raw_root = Path("Data/RAW_test")

debug_files = [
    "5cf3aee6d9001d2064299346_Human response to increased temperatures in offices (translated Humane respons til forøget temperatur i kontormiljø).pdf",
    "5cee68eed9001d2064299318_Deep Reinforcement Learning Applying an actor-critic network to a search and rescue robotics setting (translated Deep Re.pdf",
]

lot_heading = re.compile(
    r"^\s*(?:[IVXLCDM]+|[A-Z]|\d+(?:\s*[\.-]\s*\d+)*)?\s*(list of tables|table list|lot|tables|tabeller|tabeloversigt|oversigt over tabeller|fortegnelse over tabeller|liste over tabeller|liste af tabeller)\s*:?\s*$",
    re.IGNORECASE
)

for f in debug_files:
    pdf_path = raw_root / f
    print("\n" + "=" * 140)
    print("FILE:", f)
    doc = fitz.open(str(pdf_path))
    heading_page = None

    for p in range(len(doc)):
        lines = [re.sub(r"\s+", " ", ln).strip() for ln in (doc[p].get_text("text") or "").splitlines()]
        if any(lot_heading.match(ln) for ln in lines if ln):
            heading_page = p
            break

    if heading_page is None:
        print("No LOT heading found")
        doc.close()
        continue

    print(f"LOT heading page (1-based): {heading_page + 1}")
    for p in range(heading_page, min(heading_page + 3, len(doc))):
        print("\n" + "-" * 60)
        print(f"PAGE {p + 1}")
        print("-" * 60)
        txt = doc[p].get_text("text") or ""
        print(txt[:5000])  # cap output
    doc.close()

## - PLOT DISTRIBUTION - choose what metric to plot

In [ ]:
# user to choose what metric to analyze
metric_choice = input("Enter metric to analyze (num_figures/num_tables): ").strip().lower()
if metric_choice == "num_figures":
    df_to_analyze = num_figures_df
    title = "Figures"
elif metric_choice == "num_tables":
    df_to_analyze = num_tables_df
    title = "Tables"
else:
    print("Invalid choice. Defaulting to num_figures.")
    df_to_analyze = num_figures_df

# summarary statistics on num_figures
print(f"\nSummary statistics for {metric_choice}:")
print(df_to_analyze[metric_choice].describe())
# barplot of num_figures distribution
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
df_to_analyze[metric_choice].value_counts().sort_index().plot(kind='bar', color='skyblue')
plt.title(f'Distribution of Number of {title} Extracted')
plt.xlabel(f'Number of {title}')
plt.ylabel('Count of PDFs')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

# display 10 larges values in for df:
print("\nTop 10 PDFs with highest {metric_choice}:")
display(df_to_analyze.sort_values(by=metric_choice, ascending=False).head(10))

## - QUICK PDF PAGE PRINTS

In [ ]:
# ==== QUICK TEST CELL ====
### ---- PRINT X PAGES OF A SINGLE PDF ---- ###
import pdfplumber
pdf_input = "Data/RAW_test/5d1c8d7bd9001d146569a4ec_Impact of probiotic tropodithietic acid (TDA) producing Phaeobacter piscinae on microbial community members related to a.pdf"
# Use existing variables from the notebook
pdf_file = Path(pdf_input) if "pdf_input" in globals() else Path(pdf_path) / pdf_file_name
# Optional fallback if relative path differs by working directory
if not pdf_file.exists():
    alt = Path("../") / pdf_file
    if alt.exists():
        pdf_file = alt
x_pages = int(input("How many pages to print? ").strip())
with pdfplumber.open(str(pdf_file)) as pdf:
    total_pages = len(pdf.pages)
    n = min(x_pages, total_pages)
    print(f"Printing {n}/{total_pages} page(s) from: {pdf_file.name}\n")
    for i in range(n):
        text = pdf.pages[i].extract_text() or "[No extractable text]"
        print(f"{'=' * 20} PAGE {i + 1} {'=' * 20}")
        print(text)
        print()

ValueError: invalid literal for int() with base 10: ''

In [ ]:
# ==== QUICK TEST CELL ====
### ---- PRINT PAGE X OF A SINGLE PDF ---- ###
import pdfplumber
pdf_input = "Data/RAW_test/5cee68eed9001d2064299318_Deep Reinforcement Learning Applying an actor-critic network to a search and rescue robotics setting (translated Deep Re.pdf"
# Keep current pdf_file logic if needed
pdf_file = Path(pdf_input) if "pdf_input" in globals() else Path(pdf_path) / pdf_file_name
if not pdf_file.exists():
    alt = Path("../") / pdf_file
    if alt.exists():
        pdf_file = alt
page_to_print = int(input("Which page to print? (1-based): ").strip())
with pdfplumber.open(str(pdf_file)) as pdf:
    total_pages = len(pdf.pages)
    if not (1 <= page_to_print <= total_pages):
        print(f"Invalid page number. Enter a value between 1 and {total_pages}.")
    else:
        i = page_to_print - 1  # convert to 0-based index
        text = pdf.pages[i].extract_text() or "[No extractable text]"
        print(f"Printing page {page_to_print}/{total_pages} from: {pdf_file.name}\n")
        print(f"{'=' * 20} PAGE {page_to_print} {'=' * 20}")
        print(text)

# ARCHIVES

## - TARGETED (specefic file) VALIDATION CHECK

In [ ]:
# ==== ARCHIVE: TARGETED VALIDATION CHECK (num_figures) ====
# HOW TO USE
# 1) Set TARGET_FILE to a filename present in validation['pdf_file'].
# 2) Ensure Cell 8 (validation table) and Cell 5 (extract_num_figures) are already run.
# 3) Run this cell.
# ---------------------------------
# HOW TO READ
# - true_num_figures: expected value from validation table.
# - pred_num_figures: model prediction from extract_num_figures.
# - abs_error: absolute difference.
# - pct_error: absolute percent error relative to true value.
# - within_10pct: True means pass under the current tolerance rule.
from pathlib import Path
# Change this filename when doing a one-file diagnostic.
TARGET_FILE = "5d1c8d6bd9001d146569a4b3_Characterising eye-motion patterns in patients with autistic spectrum disorders using machine learning (translated Karak.pdf"
true_num_figures = float(
    validation.loc[validation["pdf_file"] == TARGET_FILE, "num_figures"].iloc[0]
)
raw_root = Path("../Data/RAW_test")
if not raw_root.exists():
    raw_root = Path("Data/RAW_test")
target_pdf = raw_root / TARGET_FILE
pred_num_figures = extract_num_figures(str(target_pdf), debug=False)
abs_error = abs(pred_num_figures - true_num_figures)
pct_error = (abs_error / true_num_figures) * 100
print("target:", target_pdf.name)
print("true_num_figures:", true_num_figures)
print("pred_num_figures:", pred_num_figures)
print("abs_error:", abs_error)
print("pct_error:", round(pct_error, 2), "%")
print("within_10pct:", pct_error <= 10)

## - FIGURES DIAGNOSTICS: current production vs git-base

In [33]:
# ==== FIGURES DIAGNOSTICS: current production vs git-base ====
import json
import re
import subprocess
from pathlib import Path

import fitz
import pandas as pd

current_nb = json.loads(Path("thesis_stats_extractor.ipynb").read_text())
base_nb = json.loads(
    subprocess.check_output(
        ["git", "show", "HEAD:exstraction_more_from_pdf/thesis_stats_extractor.ipynb"],
        text=True,
    )
)

cur_src = None
base_src = None
for cell in current_nb.get("cells", []):
    if cell.get("cell_type") != "code":
        continue
    src = "\n".join(cell.get("source", []))
    if "def _create_extractor(metric_config: dict):" in src:
        cur_src = src
        break

for cell in base_nb.get("cells", []):
    if cell.get("cell_type") != "code":
        continue
    src = "\n".join(cell.get("source", []))
    if "def extract_num_figures(" in src:
        base_src = src
        break

if cur_src is None or base_src is None:
    raise RuntimeError("Could not locate current/base num_figures source")

ns_cur = {"re": re, "fitz": fitz}
exec(cur_src, ns_cur)
extract_cur = ns_cur["extract_num_figures"]

ns_base = {"re": re, "fitz": fitz}
exec(base_src, ns_base)
extract_base = ns_base["extract_num_figures"]

raw_root = Path("../Data/RAW_test")
if not raw_root.exists():
    raw_root = Path("Data/RAW_test")

rows = []
for _, row in validation[validation["num_figures"].notna()].iterrows():
    file_name = str(row["pdf_file"]).strip()
    pdf_path = raw_root / file_name
    if not pdf_path.exists():
        continue
    cur = extract_cur(str(pdf_path), debug=False)
    base = extract_base(str(pdf_path), debug=False)
    rows.append(
        {
            "pdf_file": file_name,
            "true_num_figures": row["num_figures"],
            "current_pred": cur,
            "base_pred": base,
            "delta": cur - base,
        }
    )

cmp_df = pd.DataFrame(rows)
print("num_figures current vs git-base")
print(f"- files checked: {len(cmp_df)}")
print(f"- files with deltas: {int((cmp_df['delta'] != 0).sum())}")
display(cmp_df.sort_values('delta', ascending=False))


num_figures current vs git-base
- files checked: 11
- files with deltas: 0


,pdf_file,true_num_figures,current_pred,base_pred,delta
0,5cee68eed9001d2064299318_Deep Reinforcement Le...,20.0,20,20,0
1,5cf3aee6d9001d2064299346_Human response to inc...,24.0,24,24,0
2,5d04d261d9001d010a45c03d_Comparativ analysis o...,40.0,40,40,0
3,5d1c8d6bd9001d146569a4b3_Characterising eye-mo...,46.0,46,46,0
4,5d1c8d7bd9001d146569a4ec_Impact of probiotic t...,12.0,13,13,0
5,5d1c8d83d9001d146569a4fd_Storage of heat in th...,62.0,67,67,0
6,5d1c8d92d9001d146569a522_Investigating the imp...,38.0,38,38,0
7,5d25c7e1d9001d2d454f05ac_Facilitating longitud...,13.0,13,13,0
8,5d34488fd9001d016b24698d_Vulnerability assessm...,11.0,11,11,0
9,5d3d8341d9001d32f558c139_Application-Specific ...,28.0,30,30,0
